In [4]:
"""
Script to inspect the structure of mntss/transcoder-Llama-3.2-1B
from a local cache directory.
"""

from safetensors import safe_open
from pathlib import Path
import json
import yaml

# Use your cached location
CACHED_REPO_PATH = Path("/g/data/sz65/sg9944/hugging_face_cache/hub/models--mntss--transcoder-Llama-3.2-1B/snapshots/c37a82c1ec4cea30d424850d159b17b720ce19e2/")

print("=" * 80)
print(f"Inspecting cached repo at:")
print(f"  {CACHED_REPO_PATH}")
print("=" * 80)

# Check if path exists
if not CACHED_REPO_PATH.exists():
    print(f"\n❌ Error: Path does not exist!")
    print(f"Please verify the path: {CACHED_REPO_PATH}")
    exit(1)

# List all files in the repository
print("\n📁 Files in repository:")
print("-" * 80)
files = sorted(CACHED_REPO_PATH.rglob("*"))
for file in files:
    if file.is_file():
        rel_path = file.relative_to(CACHED_REPO_PATH)
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  {rel_path} ({size_mb:.2f} MB)")

# Check for config files (both JSON and YAML)
print("\n📋 Configuration:")
print("-" * 80)

config_json = CACHED_REPO_PATH / "config.json"
config_yaml = CACHED_REPO_PATH / "config.yaml"
config_yml = CACHED_REPO_PATH / "config.yml"

config_data = None
config_file = None

if config_yaml.exists():
    config_file = config_yaml
    with open(config_yaml, 'r') as f:
        config_data = yaml.safe_load(f)
    print(f"Found config.yaml")
elif config_yml.exists():
    config_file = config_yml
    with open(config_yml, 'r') as f:
        config_data = yaml.safe_load(f)
    print(f"Found config.yml")
elif config_json.exists():
    config_file = config_json
    with open(config_json, 'r') as f:
        config_data = json.load(f)
    print(f"Found config.json")
else:
    print("No config file found (searched for config.yaml, config.yml, config.json)")

if config_data:
    print(f"\nConfig file: {config_file}")
    print("\nConfig contents:")
    print(yaml.dump(config_data, default_flow_style=False, indent=2))

# Look for safetensors files and inspect their contents
print("\n🔍 Inspecting model weights:")
print("-" * 80)


Inspecting cached repo at:
  /g/data/sz65/sg9944/hugging_face_cache/hub/models--mntss--transcoder-Llama-3.2-1B/snapshots/c37a82c1ec4cea30d424850d159b17b720ce19e2

📁 Files in repository:
--------------------------------------------------------------------------------
  .gitattributes (0.00 MB)
  README.md (0.00 MB)
  config.yaml (0.00 MB)
  features/index.json.gz (8.14 MB)
  features/layer_0.bin (999.59 MB)
  features/layer_1.bin (1005.48 MB)
  features/layer_10.bin (1031.24 MB)
  features/layer_11.bin (1031.33 MB)
  features/layer_12.bin (1035.65 MB)
  features/layer_13.bin (1040.98 MB)
  features/layer_14.bin (1037.68 MB)
  features/layer_15.bin (878.22 MB)
  features/layer_2.bin (1019.31 MB)
  features/layer_3.bin (1024.85 MB)
  features/layer_4.bin (1031.21 MB)
  features/layer_5.bin (1038.83 MB)
  features/layer_6.bin (1041.29 MB)
  features/layer_7.bin (1036.98 MB)
  features/layer_8.bin (1026.50 MB)
  features/layer_9.bin (1034.36 MB)
  layer_0.safetensors (1032.25 MB)
  layer_1.

In [ ]:

# Look for safetensors files and inspect their contents
print("\n🔍 Inspecting model weights:")
print("-" * 80)

safetensor_files = list(CACHED_REPO_PATH.glob("*.safetensors"))
print(f"\nFound {len(safetensor_files)} safetensors files:")
for sf in safetensor_files:
    print(f"  {sf.name}")

# Inspect each safetensors file
for model_file in safetensor_files:
    print(f"\n{'=' * 80}")
    print(f"📊 Detailed inspection of: {model_file.name}")
    print("=" * 80)
    
    try:
        with safe_open(str(model_file), framework="pt", device="cpu") as f:
            keys = list(f.keys())
            print(f"\nTotal tensors: {len(keys)}\n")
            
            # Group by layer
            layers = {}
            for key in keys:
                # Try to extract layer information
                parts = key.split('.')
                
                # Common patterns: layer.0, blocks.0, etc.
                if len(parts) >= 2 and any(p.isdigit() for p in parts[:3]):
                    # Find the layer identifier
                    layer_parts = []
                    for i, part in enumerate(parts):
                        layer_parts.append(part)
                        if part.isdigit():
                            break
                    layer_name = '.'.join(layer_parts)
                else:
                    layer_name = "root"
                
                if layer_name not in layers:
                    layers[layer_name] = []
                layers[layer_name].append(key)
            
            # Print organized by layer
            for layer_name in sorted(layers.keys()):
                print(f"\n{layer_name}:")
                for key in sorted(layers[layer_name]):
                    tensor = f.get_tensor(key)
                    print(f"  {key}")
                    print(f"    Shape: {tuple(tensor.shape)}")
                    print(f"    Dtype: {tensor.dtype}")
                    if tensor.numel() > 0:
                        print(f"    Min: {tensor.min():.6f}, Max: {tensor.max():.6f}, Mean: {tensor.mean():.6f}")
            
            # Look for encoder/decoder patterns
            print("\n" + "=" * 80)
            print("🔎 Analyzing tensor naming patterns:")
            print("-" * 80)
            
            # Show all unique suffixes (after removing layer prefix)
            suffixes = set()
            for key in keys:
                parts = key.split('.')
                # Remove numeric layer identifiers
                clean_parts = [p for p in parts if not p.isdigit()]
                suffix = '.'.join(clean_parts)
                suffixes.add(suffix)
            
            print("\nUnique tensor name patterns (layer numbers removed):")
            for suffix in sorted(suffixes):
                # Find an example key with this pattern
                example = next(k for k in keys if '.'.join([p for p in k.split('.') if not p.isdigit()]) == suffix)
                tensor = f.get_tensor(example)
                print(f"  {suffix}: {tuple(tensor.shape)}")
            
            # Categorize tensors
            encoder_keys = [k for k in keys if 'encoder' in k.lower() or 'encode' in k.lower() or 'W_enc' in k]
            decoder_keys = [k for k in keys if 'decoder' in k.lower() or 'decode' in k.lower() or 'W_dec' in k]
            bias_keys = [k for k in keys if 'bias' in k.lower() or k.endswith('.b') or 'b_' in k]
            
            print("\n" + "=" * 80)
            print("📦 Tensor categorization:")
            print("-" * 80)
            
            if encoder_keys:
                print(f"\n✓ Found {len(encoder_keys)} encoder-related tensors:")
                for k in sorted(encoder_keys):
                    tensor = f.get_tensor(k)
                    print(f"  {k}: {tuple(tensor.shape)}")
            
            if decoder_keys:
                print(f"\n✓ Found {len(decoder_keys)} decoder-related tensors:")
                for k in sorted(decoder_keys):
                    tensor = f.get_tensor(k)
                    print(f"  {k}: {tuple(tensor.shape)}")
            
            if bias_keys:
                print(f"\n✓ Found {len(bias_keys)} bias tensors:")
                for k in sorted(bias_keys):
                    tensor = f.get_tensor(k)
                    print(f"  {k}: {tuple(tensor.shape)}")
            
            # Show first few keys as examples
            print("\n" + "=" * 80)
            print("📋 First 10 tensor keys (as examples):")
            print("-" * 80)
            for key in sorted(keys)[:10]:
                tensor = f.get_tensor(key)
                print(f"  {key}: {tuple(tensor.shape)}")
            if len(keys) > 10:
                print(f"  ... and {len(keys) - 10} more")
                    
    except Exception as e:
        print(f"Error inspecting safetensors: {e}")
        import traceback
        traceback.print_exc()

# Try to find README for additional context
print("\n" + "=" * 80)
print("📖 README (if available):")
print("-" * 80)
readme_path = CACHED_REPO_PATH / "README.md"
if readme_path.exists():
    with open(readme_path, 'r') as f:
        readme = f.read()
    print(readme[:2000])  # First 2000 characters
    if len(readme) > 2000:
        print(f"\n... (truncated, {len(readme)} total characters)")
else:
    print("No README.md found")

print("\n" + "=" * 80)
print("✅ Inspection complete!")
print("=" * 80)

In [7]:
## Change the working dir:
import os
new_directory_path = '/g/data4/sz65/sg9944/act_study/lora-finetuning-saes/'
# os.environ["HF_HUB_OFFLINE"] = "1"
os.chdir(new_directory_path)

In [8]:
"""
Convert mntss/transcoder-Llama-3.2-1B to sparsify format.
Uses the locally cached model.
"""

import json
import yaml
from pathlib import Path
from safetensors import safe_open
from safetensors.torch import save_file

# Use your cached location
CACHED_REPO_PATH = Path("/g/data/sz65/sg9944/hugging_face_cache/hub/models--mntss--transcoder-Llama-3.2-1B/snapshots/c37a82c1ec4cea30d424850d159b17b720ce19e2/")

def load_config(repo_path: Path) -> dict:
    """Load config from YAML or JSON file."""
    config_yaml = repo_path / "config.yaml"
    config_yml = repo_path / "config.yml"
    config_json = repo_path / "config.json"
    
    if config_yaml.exists():
        with open(config_yaml, 'r') as f:
            return yaml.safe_load(f)
    elif config_yml.exists():
        with open(config_yml, 'r') as f:
            return yaml.safe_load(f)
    elif config_json.exists():
        with open(config_json, 'r') as f:
            return json.load(f)
    else:
        return {}

def convert_transcoder_to_sparsify(
    input_path: Path = CACHED_REPO_PATH,
    output_dir: str = "./sparsify_mntss_transcoder_llama321b",
):
    """
    Convert HuggingFace transcoder to sparsify format.
    
    Args:
        input_path: Path to cached HuggingFace model
        output_dir: Where to save converted files
    """
    
    if not input_path.exists():
        raise ValueError(f"Input path does not exist: {input_path}")
    
    print(f"Reading from cached repo: {input_path}")
    print("=" * 80)
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Load config if available
    hf_config = load_config(input_path)
    if hf_config:
        print("\nHuggingFace config found:")
        print(yaml.dump(hf_config, default_flow_style=False, indent=2))
    
    # Find all layer safetensors files
    safetensor_files = sorted(input_path.glob("layer_*.safetensors"))
    
    if not safetensor_files:
        raise ValueError(f"No layer_*.safetensors files found in {input_path}")
    
    print(f"\nFound {len(safetensor_files)} layer files")
    print("=" * 80)
    
    # Process each layer
    for layer_file in safetensor_files:
        # Extract layer number from filename (e.g., "layer_0.safetensors" -> "0")
        layer_num = layer_file.stem.split('_')[1]
        layer_name = f"layers.{layer_num}"
        
        print(f"\n📦 Processing {layer_file.name} -> {layer_name}")
        print("-" * 80)
        
        with safe_open(str(layer_file), framework="pt", device="cpu") as f:
            # The keys are: W_enc, W_dec, b_enc, b_dec, W_skip
            keys = list(f.keys())
            print(f"Keys in file: {keys}")
            
            # Load all tensors
            W_enc = f.get_tensor("W_enc")
            W_dec = f.get_tensor("W_dec")
            b_enc = f.get_tensor("b_enc")
            b_dec = f.get_tensor("b_dec")
            W_skip = f.get_tensor("W_skip")
            
            # Get dimensions
            num_latents, d_in = W_enc.shape
            
            print(f"\nDimensions:")
            print(f"  W_enc:  {tuple(W_enc.shape)}")
            print(f"  W_dec:  {tuple(W_dec.shape)}")
            print(f"  b_enc:  {tuple(b_enc.shape)}")
            print(f"  b_dec:  {tuple(b_dec.shape)}")
            print(f"  W_skip: {tuple(W_skip.shape)}")
            print(f"\n  d_in:           {d_in}")
            print(f"  num_latents:    {num_latents}")
            print(f"  expansion:      {num_latents / d_in:.1f}x")
            
            # Create output directory for this layer
            layer_output = output_path / layer_name
            layer_output.mkdir(parents=True, exist_ok=True)
            
            # Save in sparsify format
            # Key mapping: W_enc -> encoder.weight, b_enc -> encoder.bias
            # W_dec and b_dec stay the same, W_skip stays the same
            save_file(
                {
                    "encoder.weight": W_enc,
                    "encoder.bias": b_enc,
                    "W_dec": W_dec,
                    "b_dec": b_dec,
                    "W_skip": W_skip,
                },
                str(layer_output / "sae.safetensors")
            )
            
            # Try to extract k from HF config if available
            k = None
            if hf_config:
                k = (hf_config.get('k') or 
                     hf_config.get('top_k') or 
                     hf_config.get('num_active'))
            
            # Fallback: Use heuristic (64x expansion suggests k ~= 32-128)
            if k is None:
                k = 64  # Conservative default for 64x expansion
                print(f"  Using default k={k}")
            else:
                print(f"  Using k={k} from config")
            
            # Create sparsify config
            cfg = {
                "d_in": int(d_in),
                "num_latents": int(num_latents),
                "expansion_factor": int(num_latents // d_in),
                "k": int(k),
                "activation": "topk",
                "transcode": True,  # This is a transcoder
                "normalize_decoder": True,
                "multi_topk": False,
                "skip_connection": True,  # Model has W_skip!
            }
            
            with open(layer_output / "cfg.json", "w") as config_file:
                json.dump(cfg, config_file, indent=2)
            
            print(f"\n✅ Saved to: {layer_output}")
    
    print("\n" + "=" * 80)
    print("✅ Conversion complete!")
    print("=" * 80)
    print(f"\nOutput directory: {output_path}")
    print(f"Converted {len(safetensor_files)} layers")
    
    print(f"\n📖 Usage instructions:")
    print("=" * 80)
    print(f"\nLoad a single layer:")
    print(f"```python")
    print(f"from sparsify import SparseCoder")
    print(f"")
    print(f"sae = SparseCoder.load_from_disk('{output_path}/layers.0')")
    print(f"```")
    print(f"\nLoad all layers:")
    print(f"```python")
    print(f"from sparsify import SparseCoder")
    print(f"")
    print(f"saes = SparseCoder.load_many('{output_path}', local=True)")
    print(f"# saes will be a dict: {{'layers.0': <SAE>, 'layers.1': <SAE>, ...}}")
    print(f"```")
    print(f"\nTest with a tensor:")
    print(f"```python")
    print(f"import torch")
    print(f"")
    print(f"# Input should be shape (batch_size * seq_len, 2048)")
    print(f"x = torch.randn(32, 2048)")
    print(f"")
    print(f"# Encode")
    print(f"output = sae.encode(x)")
    print(f"print(f'Top acts: {{output.top_acts.shape}}')  # (32, k)")
    print(f"print(f'Top indices: {{output.top_indices.shape}}')  # (32, k)")
    print(f"")
    print(f"# Decode")
    print(f"reconstructed = sae.decode(output.top_acts, output.top_indices)")
    print(f"print(f'Reconstructed: {{reconstructed.shape}}')  # (32, 2048)")
    print(f"```")

if __name__ == "__main__":
    convert_transcoder_to_sparsify()

Reading from cached repo: /g/data/sz65/sg9944/hugging_face_cache/hub/models--mntss--transcoder-Llama-3.2-1B/snapshots/c37a82c1ec4cea30d424850d159b17b720ce19e2

HuggingFace config found:
feature_input_hook: hook_resid_mid
feature_output_hook: hook_mlp_out
model_kind: transcoder_set
model_name: meta-llama/Llama-3.2-1B


Found 16 layer files

📦 Processing layer_0.safetensors -> layers.0
--------------------------------------------------------------------------------
Keys in file: ['W_dec', 'W_enc', 'W_skip', 'b_dec', 'b_enc']

Dimensions:
  W_enc:  (131072, 2048)
  W_dec:  (131072, 2048)
  b_enc:  (131072,)
  b_dec:  (2048,)
  W_skip: (2048, 2048)

  d_in:           2048
  num_latents:    131072
  expansion:      64.0x
  Using default k=64

✅ Saved to: sparsify_mntss_transcoder_llama321b/layers.0

📦 Processing layer_1.safetensors -> layers.1
--------------------------------------------------------------------------------
Keys in file: ['W_dec', 'W_enc', 'W_skip', 'b_dec', 'b_enc']

Dimens

In [2]:
"""
Quick comparison script to check if converted and reference are identical.
"""

import torch
from pathlib import Path
from safetensors import safe_open

# Paths
MNTSS_CONVERTED = Path("/g/data/sz65/sg9944/act_study/lora-finetuning-saes/sparsify_mntss_transcoder_llama321b")
ELEUTHER_REFERENCE = Path("/g/data/sz65/sg9944/hugging_face_cache/hub/models--EleutherAI--skip-transcoder-Llama-3.2-1B-131k/snapshots/9d82a6a9116ce2c0bab172f1a2656d4810d0c626")

def quick_compare(layer_num: int):
    """Quick comparison of a single layer."""
    
    # Construct paths
    converted = MNTSS_CONVERTED / f"layers.{layer_num}" / "sae.safetensors"
    reference = ELEUTHER_REFERENCE / f"layers.{layer_num}" / "sae.safetensors"
    
    print(f"Comparing layer {layer_num}...")
    print(f"  Converted: {converted}")
    print(f"  Reference: {reference}")
    
    if not converted.exists():
        print(f"  ❌ Converted not found!")
        return False
    
    if not reference.exists():
        print(f"  ❌ Reference not found!")
        return False
    
    # Load and compare
    with safe_open(str(converted), framework="pt", device="cpu") as f_conv:
        with safe_open(str(reference), framework="pt", device="cpu") as f_ref:
            conv_keys = set(f_conv.keys())
            ref_keys = set(f_ref.keys())
            
            print(f"  Converted keys: {sorted(conv_keys)}")
            print(f"  Reference keys: {sorted(ref_keys)}")
            
            if conv_keys != ref_keys:
                print(f"  ❌ Keys don't match!")
                print(f"    Missing in converted: {ref_keys - conv_keys}")
                print(f"    Extra in converted: {conv_keys - ref_keys}")
                return False
            
            # Compare each tensor
            all_match = True
            for key in sorted(conv_keys):
                t_conv = f_conv.get_tensor(key)
                t_ref = f_ref.get_tensor(key)
                
                # Check if tensors are close (allowing for numerical precision)
                if torch.allclose(t_conv.float(), t_ref.float(), rtol=1e-3, atol=1e-4):
                    print(f"  ✅ {key}: MATCH")
                else:
                    max_diff = (t_conv.float() - t_ref.float()).abs().max()
                    print(f"  ❌ {key}: DIFFER (max diff: {max_diff:.2e})")
                    all_match = False
            
            return all_match

if __name__ == "__main__":
    # Update this path!
    import sys
    
    if "YOUR_SNAPSHOT_HASH" in str(ELEUTHER_REFERENCE):
        print("Please update ELEUTHER_REFERENCE with your actual cached path!")
        print("\nTo find it, run:")
        print("  ls g/data/sz65/sg9944/hugging_face_cache/hub/models--EleutherAI--skip-transcoder-Llama-3.2-1B-131k/snapshots/")
        sys.exit(1)
    
    # Compare first layer as a test
    print("Quick comparison of layer 0:")
    print("=" * 80)
    if quick_compare(0):
        print("\n✅ Layer 0 matches! Likely the rest will too.")
        print("\nRun the full compare_transcoders.py for comprehensive checking.")
    else:
        print("\n❌ Layer 0 differs! Check conversion logic.")

Quick comparison of layer 0:
Comparing layer 0...
  Converted: /g/data/sz65/sg9944/act_study/lora-finetuning-saes/sparsify_mntss_transcoder_llama321b/layers.0/sae.safetensors
  Reference: /g/data/sz65/sg9944/hugging_face_cache/hub/models--EleutherAI--skip-transcoder-Llama-3.2-1B-131k/snapshots/9d82a6a9116ce2c0bab172f1a2656d4810d0c626/layers.0/sae.safetensors
  ❌ Reference not found!

❌ Layer 0 differs! Check conversion logic.
